# End-to-End Medallion Lakehouse Architecture
### Tech Stack: Databricks, Apache Spark (PySpark), Delta Lake, Spark SQL
**Layers:**
1. **Bronze:** Raw data ingestion & Delta storage
2. **Silver:** Cleansing, schema casting & validation
3. **Gold:** Analytical aggregations & business KPIs


In [0]:
# Paste your new Volume path here:
file_path = "/Volumes/workspace/default/raw_data/online_retail.csv" 

# Read the raw CSV
raw_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# Save as Bronze Delta Table
raw_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_bronze")
    
display(spark.read.table("sales_bronze").limit(5))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom


In [0]:
from pyspark.sql.functions import col, to_date, round

# Read from the Bronze table you just created
bronze_df = spark.read.table("sales_bronze")

# Clean the data (remove empty IDs, negative quantities, fix dates, calculate total)
silver_df = bronze_df.dropna(subset=["CustomerID"]) \
    .filter(col("Quantity") > 0) \
    .filter(col("UnitPrice") > 0) \
    .withColumn("InvoiceDate", to_date(col("InvoiceDate"), "M/d/yyyy H:mm")) \
    .withColumn("TotalAmount", round(col("Quantity") * col("UnitPrice"), 2))

# Save as Silver Delta Table
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_silver")
    
display(spark.read.table("sales_silver").limit(5))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01,2.55,17850,United Kingdom,15.3
536365,71053,WHITE METAL LANTERN,6,2010-12-01,3.39,17850,United Kingdom,20.34
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01,2.75,17850,United Kingdom,22.0
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01,3.39,17850,United Kingdom,20.34
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01,3.39,17850,United Kingdom,20.34


In [0]:
%sql
CREATE OR REPLACE TABLE sales_gold AS
SELECT 
    Description AS Product_Name,
    COUNT(DISTINCT InvoiceNo) AS Total_Orders,
    SUM(Quantity) AS Total_Units_Sold,
    ROUND(SUM(TotalAmount), 2) AS Total_Revenue
FROM sales_silver
GROUP BY Description
ORDER BY Total_Revenue DESC
LIMIT 20;

SELECT * FROM sales_gold;

Product_Name,Total_Orders,Total_Units_Sold,Total_Revenue
"PAPER CRAFT , LITTLE BIRDIE",1,80995,168469.6
REGENCY CAKESTAND 3 TIER,1703,12402,142592.95
WHITE HANGING HEART T-LIGHT HOLDER,1971,36725,100448.15
JUMBO BAG RED RETROSPOT,1600,46181,85220.78
MEDIUM CERAMIC TOP STORAGE JAR,195,77916,81416.73
POSTAGE,1099,3120,77803.96
PARTY BUNTING,1379,15291,68844.33
ASSORTED COLOUR BIRD ORNAMENT,1375,35362,56580.34
Manual,253,7173,53779.93
RABBIT NIGHT LIGHT,801,27202,51346.2
